In [2]:
import pandas as pd
import requests
from io import StringIO
import warnings
warnings.filterwarnings("ignore")  # suppresses the OpenSSL warning
import time

In [ ]:
# FETCH DATA FROM API

url_base = "https://zelma.ai/api/data/3.0?state=NJ&year="
yr_text = list(range(2015, 2026)) # list of years to loop through -- starting in 2015 for simplicity

dfs = list() # empty list to store dfs for each year

for yr in yr_text:
    url = url_base + str(yr)
    max_retries = 3  # Number of retry attempts
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=30)  # Add timeout to prevent hanging
            response.raise_for_status()  # Raise an exception for bad status codes (e.g., 404, 500)
            df = pd.read_csv(StringIO(response.text), dtype=object)
            df["TestYear"] = yr  # Add a column to indicate the year of the data
            dfs.append(df)
            break  # Success, exit retry loop
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1} failed for year {yr}: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff: 1s, 2s, 4s
            else:
                print(f"Failed to fetch data for year {yr} after {max_retries} attempts.")
    time.sleep(1)  # Delay between years to be polite to the server

df = pd.concat(dfs, ignore_index=True)


Attempt 1 failed for year 2015: ("Connection broken: ConnectionResetError(54, 'Connection reset by peer')", ConnectionResetError(54, 'Connection reset by peer'))
Attempt 1 failed for year 2020: 404 Client Error: Not Found for url: https://api.zelma-api.com:443/api/data/3.0/?state=NJ&year=2020
Attempt 2 failed for year 2020: 404 Client Error: Not Found for url: https://api.zelma-api.com:443/api/data/3.0/?state=NJ&year=2020
Attempt 3 failed for year 2020: 404 Client Error: Not Found for url: https://api.zelma-api.com:443/api/data/3.0/?state=NJ&year=2020
Failed to fetch data for year 2020 after 3 attempts.
Attempt 1 failed for year 2021: 404 Client Error: Not Found for url: https://api.zelma-api.com:443/api/data/3.0/?state=NJ&year=2021
Attempt 2 failed for year 2021: 404 Client Error: Not Found for url: https://api.zelma-api.com:443/api/data/3.0/?state=NJ&year=2021
Attempt 3 failed for year 2021: 404 Client Error: Not Found for url: https://api.zelma-api.com:443/api/data/3.0/?state=NJ&yea

In [3]:
# Drop science rows

df = df[df["Subject"] != "sci"]

In [4]:

df.to_csv("nj_data_2015_2025.csv", index=False) # save as CSV

In [ ]:
# IF STARTING HERE after restarting kernel -- just read from the csv file instead of fetching from the API again

df = pd.read_csv("nj_data_2015_2025.csv", dtype=object) # read from CSV if starting here

In [ ]:
df.columns.tolist()

['State',
 'StateAbbrev',
 'StateFips',
 'SchYear',
 'DataLevel',
 'DistName',
 'SchName',
 'NCESDistrictID',
 'StateAssignedDistID',
 'NCESSchoolID',
 'StateAssignedSchID',
 'AssmtName',
 'AssmtType',
 'Subject',
 'GradeLevel',
 'StudentGroup',
 'StudentGroup_TotalTested',
 'StudentSubGroup',
 'StudentSubGroup_TotalTested',
 'Lev1_count',
 'Lev1_percent',
 'Lev2_count',
 'Lev2_percent',
 'Lev3_count',
 'Lev3_percent',
 'Lev4_count',
 'Lev4_percent',
 'Lev5_count',
 'Lev5_percent',
 'AvgScaleScore',
 'AvgSS_SD',
 'ProficiencyCriteria',
 'ProficientOrAbove_count',
 'ProficientOrAbove_percent',
 'ParticipationRate',
 'Flag_AssmtNameChange',
 'Flag_CutScoreChange_ELA',
 'Flag_CutScoreChange_math',
 'Flag_CutScoreChange_sci',
 'Flag_CutScoreChange_soc',
 'DistType',
 'DistCharter',
 'DistLocale',
 'SchType',
 'SchLevel',
 'SchVirtual',
 'CountyName',
 'CountyCode',
 'Version',
 'TestYear']

In [ ]:
# Exploration


# notice proficiencyCriteria column, with some values "levels 3-4" and some "levels 4-5"

newark_2019 = df[(df["DistName"] == "Newark Public School District") & 
   (df["TestYear"] == 2019) & 
   (df["DataLevel"] == "District") & 
   (df["StudentSubGroup"] == "All Students")]

newark_2019.groupby("Subject")["ProficiencyCriteria"].value_counts()

# OK so it's science that has levels 3-4. Let's drop science

Subject  ProficiencyCriteria
ela      Levels 4-5             6
math     Levels 4-5             6
Name: count, dtype: int64

In [ ]:
#What's meant by "AssmtType"? Check values in that column
df["AssmtType"].unique()

array(['Regular'], dtype=object)

In [ ]:
#I want to be able to see the column names again before next cell
df.columns.tolist()

['State',
 'StateAbbrev',
 'StateFips',
 'SchYear',
 'DataLevel',
 'DistName',
 'SchName',
 'NCESDistrictID',
 'StateAssignedDistID',
 'NCESSchoolID',
 'StateAssignedSchID',
 'AssmtName',
 'AssmtType',
 'Subject',
 'GradeLevel',
 'StudentGroup',
 'StudentGroup_TotalTested',
 'StudentSubGroup',
 'StudentSubGroup_TotalTested',
 'Lev1_count',
 'Lev1_percent',
 'Lev2_count',
 'Lev2_percent',
 'Lev3_count',
 'Lev3_percent',
 'Lev4_count',
 'Lev4_percent',
 'Lev5_count',
 'Lev5_percent',
 'AvgScaleScore',
 'AvgSS_SD',
 'ProficiencyCriteria',
 'ProficientOrAbove_count',
 'ProficientOrAbove_percent',
 'ParticipationRate',
 'Flag_AssmtNameChange',
 'Flag_CutScoreChange_ELA',
 'Flag_CutScoreChange_math',
 'Flag_CutScoreChange_sci',
 'Flag_CutScoreChange_soc',
 'DistType',
 'DistCharter',
 'DistLocale',
 'SchType',
 'SchLevel',
 'SchVirtual',
 'CountyName',
 'CountyCode',
 'Version',
 'TestYear']

In [6]:
# Columns to drop
drop1 = ["AssmtType", # all "regular"
           "DistLocale",
           "AvgSS_SD",
           "ProficiencyCriteria", #all levels 4-5
           "Version"]

drop2 = df.filter(like="Flag_").columns.tolist() 

drop3 = [col for col in df.columns if col.startswith("Lev")]
# drop3 = df.filter(regex = "^Lev").columns.tolist() ## Runs very slowly

cols_to_drop = drop1 + drop2 + drop3

In [7]:
df.drop(columns=cols_to_drop, inplace=True)

In [ ]:
# Repeat save command to save the cleaned version of the data
df.to_csv("nj_data_2015_2025.csv", index=False)